# MNIST Final-Configuration Results

Runs the fixed MNIST configuration across 5 seeds and regenerates the thesis figures, tables, and checkpoint diagnostics. The notebook is intentionally thin: data loading, training, diagnostics, plotting, and saving are implemented in `final_config_utils.py`, while the learning-rule updates are imported from `learning_rules_MLP.py`.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile


def is_project_root(path: Path) -> bool:
    return (
        (path / "learning_rules_MLP.py").is_file()
        and (path / "final-config-runs" / "final_config_utils.py").is_file()
    )


def project_root_candidates():
    starts = [Path.cwd()]
    for env_name in ["COLAB_PROJECT_ROOT", "PROJECT_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            starts.append(Path(value))

    common = [
        Path("/content/colab-folder"),
        Path("/content/backprop-alternatives"),
        Path("/content/drive/MyDrive/colab-folder"),
        Path("/content/drive/MyDrive/backprop-alternatives"),
        Path("/content/drive/MyDrive/Master/colab-folder"),
    ]
    starts.extend(common)

    seen = set()
    for start in starts:
        try:
            resolved = start.expanduser().resolve()
        except Exception:
            continue
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_project_root(search_drive: bool = False) -> Path | None:
    for candidate in project_root_candidates():
        if is_project_root(candidate):
            return candidate

    if search_drive:
        search_roots = [Path("/content"), Path("/content/drive/MyDrive")]
        for root in search_roots:
            if not root.exists():
                continue
            for hit in root.rglob("learning_rules_MLP.py"):
                candidate = hit.parent
                if is_project_root(candidate):
                    return candidate
    return None


PROJECT_ROOT = find_project_root(search_drive=False)

if PROJECT_ROOT is None:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass
    PROJECT_ROOT = find_project_root(search_drive=True)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find colab-folder. Expected learning_rules_MLP.py and "
        "final-config-runs/final_config_utils.py inside the same project folder."
    )

FINAL_CONFIG_DIR = PROJECT_ROOT / "final-config-runs"
DATA_DIR = PROJECT_ROOT / "data"

os.chdir(FINAL_CONFIG_DIR)
for path in [PROJECT_ROOT, FINAL_CONFIG_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

mpl_cache = Path(tempfile.gettempdir()) / "final_config_matplotlib_cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

from final_config_utils import run_final_config

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")
print("Device will be selected automatically by the run utility.")


## Task Configuration

This cell contains the task-specific settings: dataset arguments, architecture, seeds, final hyperparameters, and diagnostic checkpoints.

In [ ]:
TASK_CONFIG = {
    "task_key": "mnist",
    "display_name": "MNIST",
    "task_type": "classification",
    "data_loader": "load_mnist",
    "activation": "relu",
    "output_dir": "results_mnist",
    "dimensions": (28 * 28, 256, 128, 10),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "seeds": list(range(5)),
    "data_kwargs": {
        "train_limit": None,
        "test_limit": None,
        "train_eval_limit": 4000,
        "batch_size": 128,
        "eval_batch_size": 1024,
        "data_dir": str(DATA_DIR / "torchvision"),
        "seed": 0,
        "mean_center_only": True,
        "num_workers": 0,
    },
    "run_epochs": 20 * 20,
    "run_print_every_epoch": 25,
    "run_configs": {
        "bp": {"lr": 0.5},
        "np": {"lr": 0.12 * 0.425, "sigma": 0.0325},
        "np_fan_in": {"lr": 0.0325 * 0.375, "sigma": 0.00875},
        "np_fixed": {"lr": 0.0225 * 0.25, "sigma": 0.12},
        "wp": {"lr": 0.015 * 0.25, "sigma": 0.0275},
    },
    "analysis": {
        "epochs": 20,
        "bp_lr": 0.2,
        "num_perturbations": 100,
        "batch_size": 128,
        "checkpoint_epochs": [1, 10, 20],
        "method_sigmas": {
            "np": 0.0325,
            "np_fan_in": 0.00875,
            "np_fixed": 0.12,
            "wp": 0.0275,
        },
    },
}

TASK_CONFIG

## Run Final Configuration

This cell performs the full multi-seed run, computes checkpoint diagnostics, saves CSV/LaTeX tables, writes PDF figures, and creates a zip archive of the output folder.

In [ ]:
outputs = run_final_config(TASK_CONFIG, project_root=PROJECT_ROOT)

print(f"Archive: {outputs['archive_path']}")
outputs['table_summary_df']

## Optional Colab Download

Uncomment the lines below in Colab if you want the generated archive to download immediately.

In [ ]:
# from google.colab import files
# files.download(str(outputs['archive_path']))